In [ ]:
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get('HUGGIE_TOKEN')

# Load only the train split, as the dataset does not have validation/test splits directly
original_train_dataset = load_dataset("Anupam007/indian-recipe-dataset", split="train", token=HF_TOKEN)

# Split the original train dataset into 80% for training and 20% for temp_test_val
train_val_test_split = original_train_dataset.train_test_split(test_size=0.2, seed=42)

# The 'train' part of this split is our actual training set
dataset_train = train_val_test_split['train']

# The 'test' part of this split (temp_test_val) will be further split into validation and test
temp_test_val = train_val_test_split['test']

# Split temp_test_val into two equal halves for validation and test (each 10% of original)
val_test_split = temp_test_val.train_test_split(test_size=0.5, seed=42)

dataset_val = val_test_split['train']
dataset_test = val_test_split['test']

print(f"Training dataset size: {len(dataset_train)}")
print(f"Validation dataset size: {len(dataset_val)}")
print(f"Test dataset size: {len(dataset_test)}")

print(f"\nFirst record of training dataset: {dataset_train[0]= }")
print(f"First record of validation dataset: {dataset_val[0]= }")
print(f"First record of test dataset: {dataset_test[0]= }")
print("Successfully loaded data from huggie face")

Cleaned_Indian_Food_Dataset.csv:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5938 [00:00<?, ? examples/s]

Training dataset size: 4750
Validation dataset size: 594
Test dataset size: 594

First record of training dataset: dataset_train[0]= {'TranslatedRecipeName': 'Shrimp Stir Fry With Cheesy Dip Mayo Recipe', 'TranslatedIngredients': '1/2 Green Bell Pepper (Capsicum) - julienned,1/2 Yellow Bell Pepper (Capsicum) - julienned,2 tablespoon Del Monte Cheesy Dip,1/2 Red Bell pepper (Capsicum) - julienned,1/2 cup Broccoli - chopped florets,15 Shrimps - de-veined,1 teaspoon Red Chilli powder,7 Garlic - minced,1 teaspoon Soy sauce,1 Onion - finely chopped,1/2 tablespoon Garam masala powder,1/2 teaspoon Salt,2 Tomatoes - finely chopped,1 tablespoon Green Chilli Sauce,2 tablespoon Sunflower Oil,4 Basil leaves - torn', 'TotalTimeInMins': 60, 'Cuisine': 'Asian', 'TranslatedInstructions': 'To prepare Shrimp Stir Fry With Cheesy Dip Mayo Recipe, de-vein the shrimps.\nWash and marinate them with the ingredients mentioned for marination.\nKeep them aside for 30 minutes.Heat a wok/kadai and add oil.\nOnce 

In [ ]:
# write a function to format the input data
def format_input_data(data):
 # print('inside format input data')
  formatted_input_data = (
      f"Recipe: {data['TranslatedRecipeName']}\n"
      f"Ingredients: {data['TranslatedIngredients']}\n"
      f"Instructions: {data['TranslatedInstructions']}\n"
      f"<|endoftext|>"
  )
  #print('after formatting input data', formatted_input_data)
  return {"text": formatted_input_data}


In [ ]:
# First, apply the formatting function to create the 'text' column.
# The original columns are still present at this stage.
dataset_train = dataset_train.map(format_input_data)
dataset_val = dataset_val.map(format_input_data)
dataset_test = dataset_test.map(format_input_data)

print("After formatting data and before dropping columns for train:", dataset_train[0])

# Now, define the columns to drop for each dataset, which are all columns except 'text'
# Since the format_input_data only returns 'text', we want to remove all original columns.
drop_columns = [col for col in dataset_train.column_names if col != 'text']

# Finally, remove the unwanted columns from all datasets.
dataset_train = dataset_train.remove_columns(drop_columns)
dataset_val = dataset_val.remove_columns(drop_columns)
dataset_test = dataset_test.remove_columns(drop_columns)

print("After dropping columns for training data:", dataset_train[0])
print("After dropping columns for validation data:", dataset_val[0])
print("After dropping columns for test data:", dataset_test[0])

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/594 [00:00<?, ? examples/s]

Map:   0%|          | 0/594 [00:00<?, ? examples/s]

After formatting data and before dropping columns for train: {'TranslatedRecipeName': 'Shrimp Stir Fry With Cheesy Dip Mayo Recipe', 'TranslatedIngredients': '1/2 Green Bell Pepper (Capsicum) - julienned,1/2 Yellow Bell Pepper (Capsicum) - julienned,2 tablespoon Del Monte Cheesy Dip,1/2 Red Bell pepper (Capsicum) - julienned,1/2 cup Broccoli - chopped florets,15 Shrimps - de-veined,1 teaspoon Red Chilli powder,7 Garlic - minced,1 teaspoon Soy sauce,1 Onion - finely chopped,1/2 tablespoon Garam masala powder,1/2 teaspoon Salt,2 Tomatoes - finely chopped,1 tablespoon Green Chilli Sauce,2 tablespoon Sunflower Oil,4 Basil leaves - torn', 'TotalTimeInMins': 60, 'Cuisine': 'Asian', 'TranslatedInstructions': 'To prepare Shrimp Stir Fry With Cheesy Dip Mayo Recipe, de-vein the shrimps.\nWash and marinate them with the ingredients mentioned for marination.\nKeep them aside for 30 minutes.Heat a wok/kadai and add oil.\nOnce the oil is heated, add chopped onion and fry till onion turns translucen

In [ ]:
def load_tokenizer(train_data):
  return tokenizer(train_data['text'], truncation=True, max_length=512, padding="max_length")

In [ ]:
from transformers import AutoTokenizer

# we load tokenizer based on the llm each llm have their own tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2", token=HF_TOKEN)

# Set the padding token
tokenizer.pad_token = tokenizer.eos_token

tokenized_train_dataset = dataset_train.map(load_tokenizer, batched=True, remove_columns=['text'])
tokenized_val_dataset = dataset_val.map(load_tokenizer, batched=True, remove_columns=['text'])
#dataset_test.map(load_tokenizer, batched=True, remove_columns=['text'])

print("before tokenization: ", dataset_train[0])
print("after tokenization: ", tokenized_train_dataset[0])

print("before tokenization: ", dataset_val[0])
print("after tokenization: ", tokenized_val_dataset[0])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/594 [00:00<?, ? examples/s]

before tokenization:  {'text': 'Recipe: Shrimp Stir Fry With Cheesy Dip Mayo Recipe\nIngredients: 1/2 Green Bell Pepper (Capsicum) - julienned,1/2 Yellow Bell Pepper (Capsicum) - julienned,2 tablespoon Del Monte Cheesy Dip,1/2 Red Bell pepper (Capsicum) - julienned,1/2 cup Broccoli - chopped florets,15 Shrimps - de-veined,1 teaspoon Red Chilli powder,7 Garlic - minced,1 teaspoon Soy sauce,1 Onion - finely chopped,1/2 tablespoon Garam masala powder,1/2 teaspoon Salt,2 Tomatoes - finely chopped,1 tablespoon Green Chilli Sauce,2 tablespoon Sunflower Oil,4 Basil leaves - torn\nInstructions: To prepare Shrimp Stir Fry With Cheesy Dip Mayo Recipe, de-vein the shrimps.\nWash and marinate them with the ingredients mentioned for marination.\nKeep them aside for 30 minutes.Heat a wok/kadai and add oil.\nOnce the oil is heated, add chopped onion and fry till onion turns translucent.\nAdd garlic and saute till the raw smell of garlic goes away.Add tomato and fry till tomatoes turns mushy and cooke

In [ ]:
# load gpt-2 model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained('gpt2', token=HF_TOKEN)
print(f"{model.config= }")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model.config= GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 50257
}



In [ ]:
from transformers import TrainingArguments
import os

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="./recipe-gpt2",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [11]:
from transformers import Trainer

trainer = Trainer(model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=data_collator)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.624700,1.635821
2,1.566622,1.601596
3,1.567177,1.585261


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=3564, training_loss=1.570819652976947, metrics={'train_runtime': 1246.5587, 'train_samples_per_second': 11.431, 'train_steps_per_second': 2.859, 'total_flos': 3723411456000000.0, 'train_loss': 1.570819652976947, 'epoch': 3.0})

In [12]:
model.save_pretrained("./recipe-gpt2-final")
tokenizer.save_pretrained("./recipe-gpt2-final")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [13]:
from transformers import pipeline

generator = pipeline("text-generation", model="./recipe-gpt2-final", tokenizer="./recipe-gpt2-final")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:
result = generator(
    "Recipe: milk-based pudding\nIngredients:",
    max_new_tokens=400,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)
print(result[0]["generated_text"])


Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_return_sequences', 'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=400) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Recipe: milk-based pudding
Ingredients: 2 teaspoons Dessicated Coconut,1/2 cup Fresh coconut - grated,1/2 teaspoon Turmeric powder (Haldi),1/4 teaspoon Cumin powder (Jeera),1/2 teaspoon Red Chilli powder,3 cups Milk,2 tablespoons Sunflower Oil,2 teaspoon Lemon juice,1/4 teaspoon Asafoetida (hing),1/4 teaspoon Methi Seeds (Fenugreek Seeds),1/4 teaspoon Black pepper powder,1 cup Water,1/4 teaspoon Salt
Instructions: To begin making the Milk-based pudding, firstly in a blender add a tablespoon of milk, salt and 1/4 teaspoon turmeric powder.
Blend till smooth.
In a separate saucepan add coconut, water, 1/2 teaspoon salt, milk and stir until milk comes to a boil.
Add water and mix well.
Let it boil for about 5 to 10 minutes.
Keep stirring occasionally.
Now gradually add lemon juice and mix well.
Add sugar and mix well.
Add the chopped coriander leaves to the mixture and let it cook for 5 minutes till it gets a nice aroma.
Turn off the heat and keep it aside to cool down.
Meanwhile, heat oil